# Project 2 — ShopSmart: build the recommender

Run cells top to bottom. Every step calls the same `src/` functions that `python -m src.train` and the API use, so the notebook and production code cannot drift apart.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.data import load_data

users, products, events = load_data()
print("users:", users.shape)
print("products:", products.shape)
print("events:", events.shape)

## Inspect the events
Do not build a model on data you have not inspected.

In [ ]:
print(events.head())
print(events["event_type"].value_counts())
print(events[["user_id", "item_id", "event_type"]].isna().sum())

## Event weighting (preprocessing, not the model)
The weight is derived from `event_type` in `src/config.py` (`view=1, cart=3, purchase=6`). These are tunable assumptions.

In [ ]:
from src.preprocess import prepare_events

events = prepare_events(events)
events[["event_type", "weight"]].drop_duplicates().sort_values("event_type")

In [ ]:
print(events.groupby("event_type")["weight"].agg(["count", "mean", "sum"]))

## User-item matrix
Repeated events for the same user and product are summed. Rows are users, columns are products.

In [ ]:
from src.preprocess import build_user_item, matrix_sparsity

matrix = build_user_item(events)
print(matrix.shape)
matrix.iloc[:5, :10]

In [ ]:
print("sparsity:", round(matrix_sparsity(matrix), 4))

## Item-item similarity (the algorithm)
Cosine similarity between product columns: two products are similar when the same users interact with them strongly.

In [ ]:
from src.recommender import ItemItemRecommender

model = ItemItemRecommender().fit(matrix)
print("item similarity shape:", model.item_similarity.shape)
print(model.item_ids[:10])

## Candidate scoring, seen-item filtering and Top-K
`score = user_vector @ item_similarity`, then items the user already interacted with are removed.

In [ ]:
user_id = matrix.index[0]
recommendations = model.recommend(user_id, k=10, explain=True)
pd.DataFrame(recommendations)

## Join product metadata
The model returns IDs and scores; the catalog supplies what a customer sees.

In [ ]:
pd.DataFrame(recommendations).merge(products, on="item_id", how="left")[
    ["item_id", "name", "category", "price", "score", "because_item_ids"]
]

## Unknown user / cold start
A collaborative model has nothing to say about an unseen user, so we fall back to popularity.

In [ ]:
from src.recommender import popular_items

print(model.recommend("unknown-user"))
pd.DataFrame(popular_items(events, k=10)).merge(products, on="item_id", how="left")[
    ["item_id", "name", "category", "score"]
]

## Evaluation
Leave-one-positive-out: hide each user's latest cart/purchase, train on the rest, check whether it comes back in the Top-10. Compare against the popularity baseline.

In [ ]:
from src.evaluate import evaluate

metrics = evaluate(events, k=10)
pd.DataFrame({"item_item": metrics["item_item"], "popularity": metrics["popularity_baseline"]})

## Experiments
New interaction and new item (sections 14–15).

In [ ]:
from src.data import load_data as reload_data
from src.experiments import new_interaction_experiment, new_item_experiment

_, _, raw_events = reload_data()
new_interaction_experiment(raw_events, products)
new_item_experiment(raw_events)

## Save the artifact
Same as running `python -m src.train` from backend/.

In [ ]:
from src.train import train

_, metadata = train()
metadata